In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import json

dataset_path = "/content/drive/MyDrive/sft_audio_dsp/formatted_sft_dataset.jsonl"

pairs = []
with open(dataset_path) as f:
    for line in f:
        if line.strip():
            pairs.append(json.loads(line))

print(f" Loaded {len(pairs)} training examples")
print("\nSample:")
print(pairs[0]["text"][:300])

 Loaded 54 training examples

Sample:
<|im_start|>system
You are an expert C++ Digital Signal Processing (DSP) programming assistant. Help users understand and implement audio DSP code in C++.
<|im_end|>
<|im_start|>user
How do I implement a low-pass filter in C++?
<|im_end|>
<|im_start|>assistant
float lowPassFilter(float input, float&


In [3]:
# Install  required libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-aqeug7mm/unsloth_823968b16875493a8bcc008d3229a78c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-aqeug7mm/unsloth_823968b16875493a8bcc008d3229a78c
  Resolved https://github.com/unslothai/unsloth.git to commit 848ede3d57167ae944ff4616eb0d56f174be87de
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 16.5 MB/s eta 0:00:0

In [5]:


# Install remaining dependencies without xformers
!pip install --no-deps trl peft accelerate bitsandbytes

In [6]:
import torch
from unsloth import FastLanguageModel

print(f"PyTorch     : {torch.__version__}")
print(f" CUDA        : {torch.cuda.is_available()}")
print(f" GPU         : {torch.cuda.get_device_name(0)}")
print(f"Unsloth     : OK")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch     : 2.10.0+cu128
 CUDA        : True
 GPU         : Tesla T4
Unsloth     : OK


In [7]:
from unsloth import FastLanguageModel
import torch

# ── Model Configuration ────────────────────────────────────────────────────────
max_seq_length = 2048  # Can increase if you have more GPU memory
dtype          = None  # Auto-detect (float16 for T4)
load_in_4bit   = True  # 4-bit quantization — essential for T4 GPU

# Load Qwen2.5-Coder base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)

print("    Model loaded successfully!")
print(f"   Model : Qwen2.5-Coder-1.5B-Instruct")
print(f"   GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

    Model loaded successfully!
   Model : Qwen2.5-Coder-1.5B-Instruct
   GPU memory used: 1.19 GB


In [8]:
# ── LoRA Configuration ─────────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,      # LoRA rank — higher = more capacity
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = 32,      # Scaling factor (usually 2x rank)
    lora_dropout        = 0.05,    # Dropout for regularization
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # Saves memory
    random_state        = 42,
)

print("LoRA adapters configured!")
print(model.print_trainable_parameters())

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA adapters configured!
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
None


In [9]:
import json
from datasets import Dataset

# ── Load Dataset ───────────────────────────────────────────────────────────────
dataset_path = "/content/drive/MyDrive/sft_audio_dsp/formatted_sft_dataset.jsonl"

pairs = []
with open(dataset_path) as f:
    for line in f:
        line = line.strip()
        if line:
            pairs.append(json.loads(line))

# Convert to HuggingFace Dataset format
dataset = Dataset.from_list(pairs)

print(f" Dataset loaded!")
print(f"   Total examples : {len(dataset)}")
print(f"   Features       : {dataset.features}")
print(f"\nSample text (first 200 chars):")
print(dataset[0]["text"][:200])

 Dataset loaded!
   Total examples : 54
   Features       : {'text': Value('string')}

Sample text (first 200 chars):
<|im_start|>system
You are an expert C++ Digital Signal Processing (DSP) programming assistant. Help users understand and implement audio DSP code in C++.
<|im_end|>
<|im_start|>user
How do I implemen


In [10]:
# ── Tokenize ───────────────────────────────────────────────────────────────────
def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation    = True,
        max_length    = 2048,
        padding       = False,
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

print(f" Dataset tokenized!")
print(f"   Columns : {tokenized_dataset.column_names}")

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

 Dataset tokenized!
   Columns : ['text', 'input_ids', 'attention_mask']


In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments

# ── Trainer Configuration ──────────────────────────────────────────────────────
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length     = 2048,
    args = TrainingArguments(
        output_dir             = "/content/drive/MyDrive/sft_audio_dsp/checkpoints",
        num_train_epochs       = 3,        # 3 passes through the dataset
        per_device_train_batch_size = 2,   # 2 samples per step (safe for T4)
        gradient_accumulation_steps = 4,   # Effective batch size = 8
        warmup_steps           = 5,
        learning_rate          = 2e-4,
        fp16                   = True,     # Use float16 on T4
        logging_steps          = 5,        # Print loss every 5 steps
        save_steps             = 20,       # Save checkpoint every 20 steps
        save_total_limit       = 2,        # Keep only last 2 checkpoints
        optim                  = "adamw_8bit",  # Memory efficient optimizer
        weight_decay           = 0.01,
        lr_scheduler_type      = "linear",
        seed                   = 42,
        report_to              = "none",   # Disable wandb
    ),
)

print(" Trainer configured!")
print(f"   Epochs          : 3")
print(f"   Batch size      : 2")
print(f"   Effective batch : 8 (with gradient accumulation)")
print(f"   Learning rate   : 2e-4")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/54 [00:00<?, ? examples/s]

 Trainer configured!
   Epochs          : 3
   Batch size      : 2
   Effective batch : 8 (with gradient accumulation)
   Learning rate   : 2e-4


In [13]:
# ── Start Training ─────────────────────────────────────────────────────────────
print(" Starting training...\n")

trainer_stats = trainer.train()

print("\n Training complete!")
print(f"   Training loss  : {trainer_stats.training_loss:.4f}")
print(f"   Total steps    : {trainer_stats.global_step}")
print(f"   Time taken     : {trainer_stats.metrics['train_runtime']:.0f} seconds")

 Starting training...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 54 | Num Epochs = 3 | Total steps = 21
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
5,0.743899
10,0.561629
15,0.493242
20,0.402491



 Training complete!
   Training loss  : 0.5536
   Total steps    : 21
   Time taken     : 62 seconds


In [15]:
# ── Save Model ─────────────────────────────────────────────────
save_path = "/content/drive/MyDrive/sft_audio_dsp/fine_tuned_model"

print(" Saving model...")

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f" Model saved to:\n   {save_path}")
print(f"\nFiles saved:")
import os
for f in os.listdir(save_path):
    size = os.path.getsize(os.path.join(save_path, f))
    print(f"   {f:40s} {size/1e6:.1f} MB")

 Saving model...
 Model saved to:
   /content/drive/MyDrive/sft_audio_dsp/fine_tuned_model

Files saved:
   README.md                                0.0 MB
   adapter_model.safetensors                73.9 MB
   adapter_config.json                      0.0 MB
   chat_template.jinja                      0.0 MB
   tokenizer_config.json                    0.0 MB
   tokenizer.json                           11.4 MB


In [16]:
# ── Inference Test ─────────────────────────────────────────────────────────────
from unsloth import FastLanguageModel

# Switch model to inference mode
FastLanguageModel.for_inference(model)

def ask_model(question: str) -> str:
    prompt = (
        "<|im_start|>system\n"
        "You are an expert C++ Digital Signal Processing (DSP) programming assistant. "
        "Help users understand and implement audio DSP code in C++.\n"
        "<|im_end|>\n"
        f"<|im_start|>user\n{question}\n<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens  = 512,
        temperature     = 0.7,
        top_p           = 0.9,
        do_sample       = True,
        pad_token_id    = tokenizer.eos_token_id,
    )

    # Decode only the new tokens (not the prompt)
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response


# ── Test Questions ─────────────────────────────────────────────────────────────
questions = [
    "How do I implement a low-pass filter in C++?",
    "How do I implement an ADSR envelope in C++?",
    "How do I apply gain to an audio signal in C++?",
]

for q in questions:
    print(f" {q}")
    print("-" * 60)
    print(ask_model(q))
    print("\n" + "=" * 60 + "\n")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 How do I implement a low-pass filter in C++?
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

class LowPassFilter {
    float b0, b1, b2, a1, a2;
    float x1 = 0, x2 = 0, y1 = 0, y2 = 0;

public:
    // First-order low-pass filter with RC cutoff.
    float process(float input) {
        float output = b0 * input + b1 * x1 + b2 * x2 - a1 * y1 - a2 * y2;
        x2 = x1; x1 = input;
        y2 = y1; y1 = output;
        return output;
    }

    void setCutOff(float freq, float sampleRate) {
        float rc = 1 / (2 * 3.14159f * freq);
        b0 = b1 = b2 = 0.0f;
        a1 = -rc / (sampleRate * 2.0f);
        a2 = rc / (sampleRate * 4.0f);
        if (freq == 0.0f)
            b0 = 1.0f / (2.0f * rc);
    }
};

// This implements a first-order RC low-pass filter. Use `setCutOff` to change cutoff frequency.



 How do I implement an ADSR envelope in C++?
------------------------------------------------------------


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


struct ADLRAudioEffect {
    struct State {
        float attackPos = 0.0f;
        float decayPos  = 0.0f;
        float sustain    = 1.0f;
        float releasePos = 0.0f;

        float sampleRate;
        float attackCoeff, decayCoeff, releaseCoeff;

        void setSampleRate(float sampleRate) {
            this->sampleRate = sampleRate;
            attackCoeff      = expf(-1.0f / (attackPos * sampleRate));
            decayCoeff       = expf(-1.0f / (decayPos  * sampleRate));
            releaseCoeff     = expf(-1.0f / (releasePos * sampleRate));
        }
    } state;

    void processADSR(const float& level, float& output) {
        if (level > 0.0f && state.attackPos < 1.0f) {
            float t = state.attackPos;
            state.attackPos += state.coeff * t;
            output           = t * level + (1.0f - t) * state.sustain;
        } else if (state.attackPos == 1.0f && state.decayPos < 1.0f) {
            float t = state.decayPos;
            state.decayPos += state.co